# LeetCode 953 — Verifying an Alien Dictionary bằng Python

**Link đề:** https://leetcode.com/problems/verifying-an-alien-dictionary/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

```python
class Solution:
    def isAlienSorted(self, words: list[str], order: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu chính xác **thứ tự từ điển** (lexicographical order): so sánh từng ký tự, và quy tắc "tiền tố ngắn hơn đứng trước".
2. Biết dùng **dict** để đổi một bảng chữ cái lạ thành **hạng** (rank) của từng ký tự.
3. Biết rằng một dãy đã sắp xếp chỉ cần kiểm tra **các cặp kề nhau** — không cần so mọi cặp.
4. Viết 4 cách: so mọi cặp (baseline), so cặp kề thủ công, "dịch" sang bảng chữ cái thường, dùng `key` + so sánh list.
5. Trace, chứng minh tính đúng, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Ở một ngôn ngữ ngoài hành tinh, người ta cũng dùng 26 chữ cái thường tiếng Anh, nhưng **thứ tự** bảng chữ cái khác. `order` là một hoán vị của 26 chữ cái, cho biết chữ nào đứng trước.

Cho danh sách `words` và `order`, trả về `True` **khi và chỉ khi** `words` đã được sắp xếp tăng dần (cho phép bằng nhau) theo thứ tự từ điển của ngôn ngữ đó.

| words | order | Kết quả | Vì sao |
|---|---|:---:|---|
| `["hello","leetcode"]` | `"hlabcdefgijkmnopqrstuvwxyz"` | `True` | `'h'` đứng trước `'l'` trong `order` |
| `["word","world","row"]` | `"worldabcefghijkmnpqstuvxyz"` | `False` | ký tự khác đầu tiên của `word`/`world` là `'d'` vs `'l'`; `'d'` đứng **sau** `'l'` |
| `["apple","app"]` | bảng chữ cái thường | `False` | `"app"` là tiền tố của `"apple"` nên `"app"` phải đứng trước |

**Constraints:**
- `1 <= len(words) <= 100`, `1 <= len(words[i]) <= 20`
- `len(order) == 26`, mọi ký tự đều là chữ thường tiếng Anh.

In [1]:
words1, order1 = ["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz"
words2, order2 = ["word", "world", "row"], "worldabcefghijkmnpqstuvxyz"
words3, order3 = ["apple", "app"], "abcdefghijklmnopqrstuvwxyz"
for w, o in [(words1, order1), (words2, order2), (words3, order3)]:
    print(w, "| order bắt đầu bằng:", o[:6], "...")

['hello', 'leetcode'] | order bắt đầu bằng: hlabcd ...
['word', 'world', 'row'] | order bắt đầu bằng: worlda ...
['apple', 'app'] | order bắt đầu bằng: abcdef ...


## 2. Nền tảng: thứ tự từ điển là gì?

Hai từ `a` và `b` được so như sau (giống cách tra từ điển giấy):

1. Đi từ trái sang phải, tìm **vị trí đầu tiên** `i` mà `a[i] != b[i]`.
2. Nếu tìm thấy: từ nào có ký tự **nhỏ hơn** tại `i` thì đứng trước. Các ký tự **sau** `i` không còn quan trọng.
3. Nếu không tìm thấy (một từ là **tiền tố** của từ kia): từ **ngắn hơn** đứng trước. Đề gọi đó là ký tự trống `∅` nhỏ hơn mọi ký tự.

Python đã so chuỗi theo đúng quy tắc này, nhưng với bảng chữ cái **thường** (theo mã Unicode). Hãy quan sát:

In [2]:
print("apple" < "banana")   # 'a' < 'b' ở vị trí 0
print("word" < "world")     # vị trí 3: 'd' < 'l' (bảng chữ thường)
print("app" < "apple")      # "app" là tiền tố -> ngắn hơn đứng trước
print("apple" < "app")      # ngược lại -> False
print("abc" <= "abc")       # bằng nhau vẫn hợp lệ khi dùng <=

True
True
True
False
True


Nhận xét: với bảng chữ cái thường thì `word < world`. Nhưng ở Ví dụ 2, `order` cho `'l'` đứng **trước** `'d'`, nên ở ngôn ngữ ngoài hành tinh `word > world`. Ta **không thể** dùng thẳng `<` của Python — phải tự định nghĩa thứ tự.

### 2.1. `enumerate` — vừa lấy chỉ số vừa lấy phần tử

`enumerate(s)` sinh ra các cặp `(chỉ_số, phần_tử)`. Rất tiện để biết mỗi chữ cái nằm ở vị trí nào trong `order`.

In [3]:
for i, ch in enumerate("hla"):
    print(i, ch)

0 h
1 l
2 a


### 2.2. `dict` — bảng tra hạng (rank) của ký tự

Ta muốn hỏi nhanh: "chữ `'l'` đứng thứ mấy trong bảng chữ cái lạ?". Một `dict` ánh xạ **ký tự → vị trí** trả lời câu đó trong O(1) trung bình.

- Tạo: `rank = {}` rồi `rank[ch] = i`.
- Viết gọn bằng **dict comprehension**: `{ch: i for i, ch in enumerate(order)}`.
- Tra: `rank['l']`.

So với `order.index('l')` (phải quét chuỗi, O(26)), tra dict nhanh và rõ ý hơn.

In [4]:
order = "hlabcdefgijkmnopqrstuvwxyz"
rank = {ch: i for i, ch in enumerate(order)}
print("rank['h'] =", rank['h'])
print("rank['l'] =", rank['l'])
print("rank['a'] =", rank['a'])
print("order.index('l') =", order.index('l'), "(cùng kết quả nhưng phải quét chuỗi)")

rank['h'] = 0
rank['l'] = 1
rank['a'] = 2
order.index('l') = 1 (cùng kết quả nhưng phải quét chuỗi)


### 2.3. `zip` — đi song song hai dãy

`zip(a, b)` ghép từng cặp phần tử cùng vị trí và **dừng ở dãy ngắn hơn**. Ta dùng nó cho hai việc:

- Đi song song hai từ: `zip("word", "world")` → `('w','w'), ('o','o'), ('r','r'), ('d','l')`.
- Lấy các **cặp kề nhau** trong list: `zip(words, words[1:])`.

`words[1:]` là **slicing**: lấy từ chỉ số 1 đến hết (bỏ phần tử đầu).

In [5]:
print(list(zip("word", "world")))
words = ["word", "world", "row"]
print("words[1:] =", words[1:])
print("cặp kề:", list(zip(words, words[1:])))

[('w', 'w'), ('o', 'o'), ('r', 'r'), ('d', 'l')]
words[1:] = ['world', 'row']
cặp kề: [('word', 'world'), ('world', 'row')]


### 2.4. Python so sánh **list** cũng theo thứ tự từ điển

`[7, 1, 2] < [7, 1, 5]` vì phần tử khác đầu tiên là `2 < 5`. Và `[1, 2] < [1, 2, 0]` vì list ngắn hơn là tiền tố. Đây là chìa khoá cho một lời giải rất gọn: biến mỗi từ thành **list các hạng** rồi để Python so list.

In [6]:
print([7, 1, 2] < [7, 1, 5])
print([1, 2] < [1, 2, 0])
print([3] < [2, 9, 9])

True
True
False


## 3. Cách trực giác (baseline) — so mọi cặp `i < j`

Định nghĩa "đã sắp xếp" một cách ngây thơ nhất: **với mọi** `i < j`, `words[i]` phải `<=` `words[j]`.

Trước hết ta cần một hàm `le(a, b, rank)` trả lời "`a <= b` theo bảng chữ cái lạ?". Viết đúng 3 bước của mục 2:

In [7]:
def le(a: str, b: str, rank: dict) -> bool:
    # So từng ký tự đến khi hết từ ngắn hơn
    for ca, cb in zip(a, b):
        if ca != cb:                      # vị trí khác đầu tiên quyết định
            return rank[ca] < rank[cb]
    # Không có vị trí khác: một từ là tiền tố của từ kia
    return len(a) <= len(b)

rank_std = {ch: i for i, ch in enumerate("abcdefghijklmnopqrstuvwxyz")}
print(le("app", "apple", rank_std))   # True
print(le("apple", "app", rank_std))   # False
rank2 = {ch: i for i, ch in enumerate("worldabcefghijkmnpqstuvxyz")}
print(le("word", "world", rank2))     # False vì 'd' sau 'l'

True
False
False


In [8]:
def is_alien_sorted_all_pairs(words: list[str], order: str) -> bool:
    rank = {ch: i for i, ch in enumerate(order)}
    n = len(words)
    for i in range(n):
        for j in range(i + 1, n):
            if not le(words[i], words[j], rank):
                return False
    return True

print(is_alien_sorted_all_pairs(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz"))
print(is_alien_sorted_all_pairs(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz"))
print(is_alien_sorted_all_pairs(["apple", "app"], "abcdefghijklmnopqrstuvwxyz"))

True
False
False


Độ phức tạp baseline: có `n(n-1)/2` cặp, mỗi cặp so tối đa `L` ký tự (`L` = độ dài từ dài nhất) → **O(n²·L)**. Với `n ≤ 100` vẫn chạy được, nhưng ta đang làm thừa rất nhiều.

## 4. Nút thắt — vì sao so mọi cặp là thừa?

Quan hệ `<=` của thứ tự từ điển có tính **bắc cầu** (transitive): nếu `a <= b` và `b <= c` thì `a <= c`.

Vậy nếu mọi **cặp kề** đều đúng thứ tự:

```text
w0 <= w1, w1 <= w2, w2 <= w3, ...
```

thì nối chuỗi lại ta có ngay `w0 <= w2`, `w0 <= w3`, ... — mọi cặp `i < j` đều tự động đúng. Ngược lại, nếu có một cặp kề sai thứ tự thì dãy chắc chắn **không** sắp xếp.

➡ Chỉ cần kiểm tra **n − 1 cặp kề**.

## 5. Insight của bài

Gộp lại, bài này chỉ gồm hai ý:

1. **Đổi bảng chữ cái** → dùng `rank = {ch: i for i, ch in enumerate(order)}` để biết chữ nào nhỏ hơn.
2. **Sắp xếp ⇔ mọi cặp kề đúng thứ tự** → duyệt `zip(words, words[1:])`.

Phần so hai từ chỉ cần nhớ: *ký tự khác đầu tiên quyết định; nếu không có thì từ ngắn hơn đứng trước*.

In [9]:
# Minh hoạ: chỉ có n-1 cặp kề cần kiểm tra
words = ["a", "b", "c", "d", "e"]
pairs = list(zip(words, words[1:]))
print("Số cặp kề:", len(pairs), "| số cặp tất cả:", len(words) * (len(words) - 1) // 2)
print(pairs)

Số cặp kề: 4 | số cặp tất cả: 10
[('a', 'b'), ('b', 'c'), ('c', 'd'), ('d', 'e')]


## 6. Lời giải chính — xây từng đoạn rồi ghép

### Đoạn 1: bảng hạng
```python
rank = {ch: i for i, ch in enumerate(order)}
```

### Đoạn 2: so một cặp kề `(a, b)`
- Duyệt song song `for ca, cb in zip(a, b)`.
- Gặp `ca != cb`: nếu `rank[ca] > rank[cb]` → sai thứ tự, trả `False` ngay. Nếu nhỏ hơn → cặp này ổn, **`break`** để sang cặp tiếp (không xét ký tự phía sau nữa!).
- Nếu vòng `for` chạy hết mà không `break` → một từ là tiền tố → sai khi `len(a) > len(b)`.

Python có cú pháp **`for ... else`**: khối `else` chạy **chỉ khi** vòng lặp kết thúc mà **không** gặp `break`. Đây đúng là trường hợp "không tìm thấy ký tự khác".

In [10]:
# Thử cú pháp for ... else
for x in [1, 2, 3]:
    if x == 2:
        print("gặp 2 -> break")
        break
else:
    print("không chạy vì đã break")

for x in [1, 3, 5]:
    if x == 2:
        break
else:
    print("vòng lặp chạy hết, không break -> vào else")

gặp 2 -> break
vòng lặp chạy hết, không break -> vào else


### Ghép lại

In [11]:
def is_alien_sorted(words: list[str], order: str) -> bool:
    rank = {ch: i for i, ch in enumerate(order)}          # Đoạn 1
    for a, b in zip(words, words[1:]):                    # chỉ cặp kề
        for ca, cb in zip(a, b):                          # Đoạn 2
            if ca != cb:
                if rank[ca] > rank[cb]:
                    return False                          # sai thứ tự
                break                                     # cặp này ổn
        else:                                             # không có ký tự khác
            if len(a) > len(b):
                return False                              # "apple" trước "app"
    return True

print(is_alien_sorted(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz"))  # True
print(is_alien_sorted(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz"))  # False
print(is_alien_sorted(["apple", "app"], "abcdefghijklmnopqrstuvwxyz"))  # False

True
False
False


## 7. Trace từng bước

Ví dụ 2: `words = ["word","world","row"]`, `order = "worldabc..."` → `rank: w=0, o=1, r=2, l=3, d=4, ...`

| Cặp | Vị trí | ca | cb | rank[ca] | rank[cb] | Quyết định |
|---|:---:|:---:|:---:|:---:|:---:|---|
| word / world | 0 | w | w | | | giống, đi tiếp |
| | 1 | o | o | | | giống |
| | 2 | r | r | | | giống |
| | 3 | d | l | 4 | 3 | 4 > 3 → **return False** |

Cặp `world / row` không bao giờ được xét vì đã trả `False`.

In [12]:
def trace_alien(words: list[str], order: str) -> bool:
    rank = {ch: i for i, ch in enumerate(order)}
    for k, (a, b) in enumerate(zip(words, words[1:])):
        print(f"Cặp {k}: {a!r} vs {b!r}")
        for pos, (ca, cb) in enumerate(zip(a, b)):
            if ca == cb:
                print(f"   pos={pos}: {ca} == {cb} -> đi tiếp")
                continue
            print(f"   pos={pos}: {ca}(rank {rank[ca]}) vs {cb}(rank {rank[cb]})", end=" ")
            if rank[ca] > rank[cb]:
                print("-> sai thứ tự, return False")
                return False
            print("-> đúng thứ tự, sang cặp sau")
            break
        else:
            print(f"   hết {min(len(a), len(b))} ký tự chung; len {len(a)} vs {len(b)}", end=" ")
            if len(a) > len(b):
                print("-> từ dài đứng trước, return False")
                return False
            print("-> ổn")
    print("Mọi cặp kề đều đúng -> return True")
    return True

trace_alien(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz")
print()
trace_alien(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz")
print()
trace_alien(["app", "apple", "apz", "b"], "abcdefghijklmnopqrstuvwxyz")

Cặp 0: 'word' vs 'world'
   pos=0: w == w -> đi tiếp
   pos=1: o == o -> đi tiếp
   pos=2: r == r -> đi tiếp
   pos=3: d(rank 4) vs l(rank 3) -> sai thứ tự, return False

Cặp 0: 'hello' vs 'leetcode'
   pos=0: h(rank 0) vs l(rank 1) -> đúng thứ tự, sang cặp sau
Mọi cặp kề đều đúng -> return True

Cặp 0: 'app' vs 'apple'
   pos=0: a == a -> đi tiếp
   pos=1: p == p -> đi tiếp
   pos=2: p == p -> đi tiếp
   hết 3 ký tự chung; len 3 vs 5 -> ổn
Cặp 1: 'apple' vs 'apz'
   pos=0: a == a -> đi tiếp
   pos=1: p == p -> đi tiếp
   pos=2: p(rank 15) vs z(rank 25) -> đúng thứ tự, sang cặp sau
Cặp 2: 'apz' vs 'b'
   pos=0: a(rank 0) vs b(rank 1) -> đúng thứ tự, sang cặp sau
Mọi cặp kề đều đúng -> return True


True

Ví dụ thứ ba cho thấy đủ ba nhánh: cặp tiền tố (`app`/`apple` → ổn), cặp quyết định ở vị trí giữa (`apple`/`apz`, `'p' < 'z'`) và cặp quyết định ngay vị trí 0.

## 8. Code nộp LeetCode

### Cách A — cặp kề + `for ... else` (lời giải chính)

In [13]:
class Solution:
    def isAlienSorted(self, words: list[str], order: str) -> bool:
        rank = {ch: i for i, ch in enumerate(order)}
        for a, b in zip(words, words[1:]):
            for ca, cb in zip(a, b):
                if ca != cb:
                    if rank[ca] > rank[cb]:
                        return False
                    break
            else:
                if len(a) > len(b):
                    return False
        return True

s = Solution()
print(s.isAlienSorted(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz"))
print(s.isAlienSorted(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz"))
print(s.isAlienSorted(["apple", "app"], "abcdefghijklmnopqrstuvwxyz"))

True
False
False


### Cách B — biến từ thành list hạng, để Python so list

Vì Python so list theo đúng thứ tự từ điển (mục 2.4), ta chỉ cần đổi `"word"` → `[0, 1, 2, 4]`. Khi đó so hai list chính là so hai từ theo bảng chữ cái lạ, kể cả quy tắc tiền tố.

In [14]:
class SolutionKey:
    def isAlienSorted(self, words: list[str], order: str) -> bool:
        rank = {ch: i for i, ch in enumerate(order)}
        keys = [[rank[ch] for ch in w] for w in words]
        return all(x <= y for x, y in zip(keys, keys[1:]))

rank = {ch: i for i, ch in enumerate("worldabcefghijkmnpqstuvxyz")}
print("word ->", [rank[c] for c in "word"], "| world ->", [rank[c] for c in "world"])
print(SolutionKey().isAlienSorted(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz"))

word -> [0, 1, 2, 4] | world -> [0, 1, 2, 3, 4]
False


`all(...)` trả `True` nếu mọi phần tử của dãy là `True` (và **dừng sớm** ở phần tử `False` đầu tiên). Biểu thức `x <= y for x, y in ...` là **generator expression** — sinh từng giá trị khi cần, không tạo list.

### Cách C — "dịch" sang bảng chữ cái thường

Một ý sáng tạo: đổi mỗi chữ lạ thành chữ thường ở **cùng hạng** (`order[i]` → `'a'+i`). Sau khi dịch, thứ tự lạ trở thành thứ tự chuẩn, và ta dùng `<=` của chuỗi Python. Công cụ: `str.maketrans` + `str.translate`.

In [15]:
import string
order = "worldabcefghijkmnpqstuvxyz"
table = str.maketrans(order, string.ascii_lowercase)
print("word  ->", "word".translate(table))
print("world ->", "world".translate(table))
print("dịch rồi so:", "word".translate(table) <= "world".translate(table))

word  -> abce
world -> abcde
dịch rồi so: False


In [16]:
import string

class SolutionTranslate:
    def isAlienSorted(self, words: list[str], order: str) -> bool:
        table = str.maketrans(order, string.ascii_lowercase)
        t = [w.translate(table) for w in words]
        return t == sorted(t)

print(SolutionTranslate().isAlienSorted(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz"))
print(SolutionTranslate().isAlienSorted(["apple", "app"], "abcdefghijklmnopqrstuvwxyz"))

True
False


Cách C dùng `sorted` nên tốn O(n log n · L); có thể thay bằng so cặp kề để về O(N). Nó vẫn đáng biết vì kỹ thuật **chuẩn hoá dữ liệu về dạng quen thuộc** dùng được ở rất nhiều bài.

## 9. Vì sao đúng?

**Bổ đề so hai từ.** Hàm so cặp trả "sai thứ tự" đúng khi `a > b` theo thứ tự lạ:
- Nếu tồn tại vị trí khác đầu tiên `i`, thứ tự từ điển được định nghĩa chỉ bởi `rank[a[i]]` và `rank[b[i]]` — ta so đúng cặp đó rồi `break`, không nhìn ký tự sau.
- Nếu không tồn tại, từ ngắn là tiền tố; `a > b` ⇔ `len(a) > len(b)`. Hai từ bằng nhau (`len` bằng) là hợp lệ.

**Invariant của vòng ngoài.** Trước khi xét cặp `(words[k], words[k+1])`, ta đã biết `words[0] <= words[1] <= ... <= words[k]`. Nhờ tính bắc cầu, dãy đến `k` đã sắp xếp. Nếu cặp mới ổn, invariant mở rộng đến `k+1`; nếu không ổn, dãy chắc chắn không sắp xếp → `False` là đúng. Hết vòng → toàn bộ dãy sắp xếp → `True`.

## 10. Độ phức tạp

Gọi `N` = tổng số ký tự của mọi từ, `n` = số từ, `L` = độ dài tối đa.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Baseline mọi cặp | O(n²·L) | O(1) (+26 cho rank) | dễ nghĩ, thừa việc |
| A. Cặp kề + for/else | **O(N)** | O(1) | mỗi ký tự được so tối đa 2 lần (với từ trước và từ sau) |
| B. List hạng + `all` | O(N) | O(N) | rất gọn, tốn bộ nhớ cho keys |
| C. Dịch + `sorted` | O(n log n · L) | O(N) | sáng tạo; thay `sorted` bằng cặp kề để về O(N) |

Bảng `rank` luôn có đúng 26 phần tử nên coi là O(1).

## 11. Bộ test

Nhóm test: ví dụ đề, một từ, từ trùng nhau, tiền tố (đúng & sai), quyết định ở vị trí cuối, thứ tự đảo ngược hoàn toàn. Cuối cùng đối chiếu tất cả các cách trên dữ liệu ngẫu nhiên.

In [17]:
ABC = "abcdefghijklmnopqrstuvwxyz"
REV = ABC[::-1]
tests = [
    (["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz", True),
    (["word", "world", "row"], "worldabcefghijkmnpqstuvxyz", False),
    (["apple", "app"], ABC, False),
    (["app", "apple"], ABC, True),          # tiền tố đúng chiều
    (["z"], ABC, True),                     # một từ
    (["aa", "aa", "aa"], ABC, True),        # từ trùng nhau
    (["abcd", "abce"], ABC, True),          # khác ở vị trí cuối
    (["abce", "abcd"], ABC, False),
    (["c", "b", "a"], REV, True),           # bảng chữ cái đảo ngược
    (["a", "b", "c"], REV, False),
    (["kuvp", "q"], "ngxlkthsjuoqcpavbfdermiywz", True),
]
impls = {
    "baseline": is_alien_sorted_all_pairs,
    "chính": is_alien_sorted,
    "LC-A": Solution().isAlienSorted,
    "LC-B": SolutionKey().isAlienSorted,
    "LC-C": SolutionTranslate().isAlienSorted,
}
for words, order, expected in tests:
    for name, f in impls.items():
        got = f(words, order)
        assert got == expected, (name, words, got, expected)
    print(f"{str(words):32} -> {expected}  PASS")
print("Tất cả test PASS!")

['hello', 'leetcode']            -> True  PASS
['word', 'world', 'row']         -> False  PASS
['apple', 'app']                 -> False  PASS
['app', 'apple']                 -> True  PASS
['z']                            -> True  PASS
['aa', 'aa', 'aa']               -> True  PASS
['abcd', 'abce']                 -> True  PASS
['abce', 'abcd']                 -> False  PASS
['c', 'b', 'a']                  -> True  PASS
['a', 'b', 'c']                  -> False  PASS
['kuvp', 'q']                    -> True  PASS
Tất cả test PASS!


In [18]:
import random

def random_case(rng):
    order = list(ABC)
    rng.shuffle(order)
    order = "".join(order)
    letters = order[:3]                      # ít chữ để dễ có tiền tố, trùng nhau
    words = ["".join(rng.choice(letters) for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 6))]
    return words, order

rng = random.Random(953)
for _ in range(3000):
    words, order = random_case(rng)
    results = {name: f(words, order) for name, f in impls.items()}
    assert len(set(results.values())) == 1, (words, order, results)
print("3000 test ngẫu nhiên: mọi cách cho cùng kết quả")

3000 test ngẫu nhiên: mọi cách cho cùng kết quả


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên `break` sau khi tìm được ký tự khác đầu tiên

In [19]:
def wrong_no_break(words, order):
    rank = {ch: i for i, ch in enumerate(order)}
    for a, b in zip(words, words[1:]):
        for ca, cb in zip(a, b):
            if ca != cb and rank[ca] > rank[cb]:
                return False          # tiếp tục xét cả ký tự phía sau -> sai!
        if len(a) > len(b):
            return False
    return True

# "ab" < "ba" vì 'a' < 'b' ở vị trí 0, nhưng vị trí 1 có 'b' > 'a'
print("Sai :", wrong_no_break(["ab", "ba"], ABC))
print("Đúng:", is_alien_sorted(["ab", "ba"], ABC))

Sai : False
Đúng: True


**Vì sao sai:** sau ký tự khác đầu tiên, các ký tự phía sau **không** còn ảnh hưởng. Code trên còn có lỗi thứ hai: kiểm tra `len(a) > len(b)` cả khi đã có ký tự khác (ví dụ `["ba", "c"]` hợp lệ nhưng bị trả `False`). **Sửa:** `break` ngay khi gặp ký tự khác và chỉ xét độ dài khi **không** có ký tự khác (`for ... else`).

In [20]:
print("Sai :", wrong_no_break(["ba", "c"], ABC))
print("Đúng:", is_alien_sorted(["ba", "c"], ABC))

Sai : False
Đúng: True


### Lỗi 2: Quên trường hợp tiền tố

Chỉ so ký tự trong `zip` mà không xét độ dài → `["apple", "app"]` bị coi là hợp lệ vì `zip` dừng sau 3 ký tự giống nhau.

In [21]:
def wrong_no_prefix(words, order):
    rank = {ch: i for i, ch in enumerate(order)}
    for a, b in zip(words, words[1:]):
        for ca, cb in zip(a, b):
            if ca != cb:
                if rank[ca] > rank[cb]:
                    return False
                break
    return True

print("Sai :", wrong_no_prefix(["apple", "app"], ABC))
print("Đúng:", is_alien_sorted(["apple", "app"], ABC))

Sai : True
Đúng: False


### Lỗi 3: Dùng thẳng `<` của Python hoặc so `ord`

`"word" < "world"` là `True` với bảng chữ thường nhưng **sai** với bảng chữ lạ. Phải so `rank[...]`, không so `ca < cb` hay `ord(ca) < ord(cb)`.

### Lỗi 4: Dùng `order.index(ch)` trong vòng lặp

Không sai kết quả, nhưng mỗi lần gọi quét lại chuỗi `order`. Thói quen tốt: tính `rank` **một lần**.

### Lỗi 5: Dùng `<` thay cho `<=` khi so list hạng

`all(x < y ...)` sẽ trả `False` cho `["aa", "aa"]`, trong khi hai từ bằng nhau vẫn là "đã sắp xếp".

## 13. Mẹo nhận dạng pattern

- Đề hỏi "dãy có **được sắp xếp** không?" → nghĩ ngay **so cặp kề** (`zip(a, a[1:])`), nhờ tính bắc cầu.
- Đề cho một **thứ tự tuỳ chỉnh** (bảng chữ cái lạ, thứ tự ưu tiên, thứ hạng) → tạo `rank = {x: i ...}` rồi so theo rank, hoặc dùng làm `key`.
- So hai chuỗi/list "theo từ điển" → nhớ quy tắc 2 nhánh: **ký tự khác đầu tiên** hoặc **tiền tố ngắn hơn**.
- Muốn dùng công cụ có sẵn với thứ tự lạ → **chuẩn hoá** (dịch về thứ tự chuẩn, hoặc đổi sang list số).

## 14. Biến thể

1. **Sắp xếp** `words` theo bảng chữ cái lạ: `sorted(words, key=lambda w: [rank[c] for c in w])`.
2. **LeetCode 1122 — Relative Sort Array:** sắp xếp theo thứ tự cho trước, phần tử ngoài danh sách xếp sau.
3. **LeetCode 791 — Custom Sort String:** sắp lại chuỗi theo `order`.
4. **LeetCode 269 — Alien Dictionary (Hard, Premium):** bài ngược — cho danh sách từ đã sắp xếp, **suy ra** `order`. Ý tưởng: mỗi cặp kề cho một cạnh "chữ `x` trước chữ `y`" từ ký tự khác đầu tiên → topological sort.

In [22]:
rank = {ch: i for i, ch in enumerate("worldabcefghijkmnpqstuvxyz")}
words = ["row", "world", "word", "dog", "low"]
print(sorted(words, key=lambda w: [rank[c] for c in w]))

# Ý tưởng của biến thể 4: cạnh suy ra từ mỗi cặp kề đã sắp xếp
sorted_words = ["wrt", "wrf", "er", "ett", "rftt"]
for a, b in zip(sorted_words, sorted_words[1:]):
    for ca, cb in zip(a, b):
        if ca != cb:
            print(f"{a} < {b}  =>  {ca} đứng trước {cb}")
            break

['world', 'word', 'row', 'low', 'dog']
wrt < wrf  =>  t đứng trước f
wrf < er  =>  w đứng trước e
er < ett  =>  r đứng trước t
ett < rftt  =>  e đứng trước r


## 15. Bài tập tự luyện (TODO)

Mỗi bài có cell TODO riêng — **không** gọi hàm khi chưa làm xong. Test gợi ý để ở dạng comment.

### Bài 1 — Bảng hạng
Viết `build_rank(order)` trả về dict ký tự → vị trí.

In [23]:
def build_rank(order: str) -> dict:
    # TODO
    pass

# assert build_rank("cba") == {'c': 0, 'b': 1, 'a': 2}

### Bài 2 — So hai từ
Viết `compare(a, b, order)` trả về `-1` nếu `a < b`, `0` nếu bằng, `1` nếu `a > b` theo bảng chữ cái lạ.

In [24]:
def compare(a: str, b: str, order: str) -> int:
    # TODO
    pass

# assert compare("word", "world", "worldabcefghijkmnpqstuvxyz") == 1
# assert compare("app", "apple", "abcdefghijklmnopqrstuvwxyz") == -1
# assert compare("abc", "abc", "abcdefghijklmnopqrstuvwxyz") == 0

### Bài 3 — Vị trí sai đầu tiên
Viết `first_bad_index(words, order)` trả về chỉ số `i` nhỏ nhất mà `words[i] > words[i+1]`, hoặc `-1` nếu đã sắp xếp.

In [25]:
def first_bad_index(words: list[str], order: str) -> int:
    # TODO
    pass

# assert first_bad_index(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz") == 0
# assert first_bad_index(["a", "b", "a"], "abcdefghijklmnopqrstuvwxyz") == 1
# assert first_bad_index(["a", "b"], "abcdefghijklmnopqrstuvwxyz") == -1

### Bài 4 — Sắp xếp theo bảng chữ cái lạ
Viết `alien_sort(words, order)` trả về list đã sắp xếp theo thứ tự lạ.

In [26]:
def alien_sort(words: list[str], order: str) -> list[str]:
    # TODO
    pass

# assert alien_sort(["b", "a", "c"], "cba" + "defghijklmnopqrstuvwxyz") == ["c", "b", "a"]

### Bài 5 — Sắp xếp giảm dần?
Viết `is_alien_sorted_desc(words, order)` kiểm tra dãy có **không tăng** (giảm dần, cho phép bằng) không.

In [27]:
def is_alien_sorted_desc(words: list[str], order: str) -> bool:
    # TODO
    pass

# assert is_alien_sorted_desc(["apple", "app"], "abcdefghijklmnopqrstuvwxyz") == True
# assert is_alien_sorted_desc(["a", "b"], "abcdefghijklmnopqrstuvwxyz") == False

### Bài 6 (nâng cao) — Suy ra các ràng buộc
Cho `words` **đã** sắp xếp theo một bảng chữ cái lạ chưa biết, trả về list các cặp `(x, y)` nghĩa là "`x` đứng trước `y`", lấy từ mỗi cặp kề (bỏ qua cặp không có ký tự khác). Giữ thứ tự xuất hiện.

In [28]:
def derive_edges(words: list[str]) -> list[tuple[str, str]]:
    # TODO
    pass

# assert derive_edges(["wrt", "wrf", "er", "ett", "rftt"]) == [('t', 'f'), ('w', 'e'), ('r', 't'), ('e', 'r')]

## 16. Gợi ý 3 mức

| Bài | Mức 1 (hướng) | Mức 2 (công cụ) | Mức 3 (gần đáp án) |
|---|---|---|---|
| 1 | mỗi ký tự cần vị trí | `enumerate(order)` | `{ch: i for i, ch in enumerate(order)}` |
| 2 | ký tự khác đầu tiên hoặc độ dài | `zip(a, b)` | không có ký tự khác → so `len(a)` với `len(b)` |
| 3 | duyệt cặp kề kèm chỉ số | `enumerate(zip(words, words[1:]))` | dùng lại `compare(...) == 1` |
| 4 | sắp xếp với `key` | `key` trả về list hạng | `sorted(words, key=lambda w: [rank[c] for c in w])` |
| 5 | đảo chiều so sánh | dùng list hạng | `all(x >= y ...)` |
| 6 | mỗi cặp kề cho tối đa 1 cạnh | ký tự khác đầu tiên | `append((ca, cb))` rồi `break` |

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [29]:
def ans_build_rank(order):
    return {ch: i for i, ch in enumerate(order)}

def ans_compare(a, b, order):
    rank = ans_build_rank(order)
    for ca, cb in zip(a, b):
        if ca != cb:
            return -1 if rank[ca] < rank[cb] else 1
    if len(a) == len(b):
        return 0
    return -1 if len(a) < len(b) else 1

def ans_first_bad_index(words, order):
    for i, (a, b) in enumerate(zip(words, words[1:])):
        if ans_compare(a, b, order) == 1:
            return i
    return -1

def ans_alien_sort(words, order):
    rank = ans_build_rank(order)
    return sorted(words, key=lambda w: [rank[c] for c in w])

def ans_is_alien_sorted_desc(words, order):
    rank = ans_build_rank(order)
    keys = [[rank[c] for c in w] for w in words]
    return all(x >= y for x, y in zip(keys, keys[1:]))

def ans_derive_edges(words):
    edges = []
    for a, b in zip(words, words[1:]):
        for ca, cb in zip(a, b):
            if ca != cb:
                edges.append((ca, cb))
                break
    return edges

assert ans_build_rank("cba") == {'c': 0, 'b': 1, 'a': 2}
assert ans_compare("word", "world", "worldabcefghijkmnpqstuvxyz") == 1
assert ans_compare("app", "apple", ABC) == -1
assert ans_compare("abc", "abc", ABC) == 0
assert ans_first_bad_index(["word", "world", "row"], "worldabcefghijkmnpqstuvxyz") == 0
assert ans_first_bad_index(["a", "b", "a"], ABC) == 1
assert ans_first_bad_index(["a", "b"], ABC) == -1
assert ans_alien_sort(["b", "a", "c"], "cba" + "defghijklmnopqrstuvwxyz") == ["c", "b", "a"]
assert ans_is_alien_sorted_desc(["apple", "app"], ABC) == True
assert ans_is_alien_sorted_desc(["a", "b"], ABC) == False
assert ans_derive_edges(["wrt", "wrf", "er", "ett", "rftt"]) == [('t', 'f'), ('w', 'e'), ('r', 't'), ('e', 'r')]
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isAlienSorted(self, words: list[str], order: str) -> bool:
        rank = {ch: i for i, ch in enumerate(order)}
        for a, b in zip(words, words[1:]):
            for ca, cb in zip(a, b):
                if ca != cb:
                    if rank[ca] > rank[cb]:
                        return False
                    break
            else:
                if len(a) > len(b):
                    return False
        return True
```

Phiên bản 3 dòng: `keys = [[rank[c] for c in w] for w in words]` rồi `return all(x <= y for x, y in zip(keys, keys[1:]))`.

**Checklist trước khi submit**
- [ ] Đã tạo `rank` từ `order` (không so `ord`/chuỗi trực tiếp)?
- [ ] Chỉ so **cặp kề**?
- [ ] Có `break` ngay sau ký tự khác đầu tiên?
- [ ] Xử lý tiền tố: `["apple","app"]` → `False`, `["app","apple"]` → `True`?
- [ ] Từ bằng nhau vẫn hợp lệ (`<=`, không phải `<`)?
- [ ] `words` chỉ có 1 phần tử → `True`?